## Training with classic machine learning models

In [1]:
import pandas as pd

snapchat_df = pd.read_json("deduped_snapchat.jsonl", lines=True)
youtube_df = pd.read_json("deduped_youtube.jsonl", lines=True)

snapchat_df["source"] = "snapchat"
youtube_df["source"] = "youtube"

df = pd.concat([snapchat_df, youtube_df], ignore_index=True)

# Flatten the body text
df["text"] = df["body"].apply(lambda x: " ".join(x))

# Use LLM for ground truth
df["label"] = df["llm_sentiment"].str.lower().str.strip()

# Map to integers
label_map = {"negative": 0, "neutral": 1, "positive": 2}
df["label_id"] = df["label"].map(label_map)

# Drop NaNs or unknowns
df = df.dropna(subset=["label_id", "text"])
print(df['label'].value_counts())

print(df.head())

label
negative    983
neutral     368
positive    157
Name: count, dtype: int64
                                               title  \
0    Should Indonesia ban social media for children?   
1  Should Indonesia ban social media for children...   
2  Social Media's Grip on Teen Lives: A Deep Dive...   
3  Australia: Under-16s banned from social media ...   
4                     Breaking the Doom-Scroll Cycle   

                                                body  \
0  [S, ocial media addiction is becoming an incre...   
1  [S, ocial media addiction is becoming an incre...   
2  [A recent report by the Pew Research Centre hi...   
3  [From November 2025, Australians under the age...   
4  [This article is written by a student writer f...   

                                            provider  \
0  {'name': 'thejakartapost.com', 'domain': 'thej...   
1  {'name': 'thejakartapost.com', 'domain': 'thej...   
2  {'name': 'devdiscourse.com', 'domain': 'devdis...   
3  {'name': 'sxm-talks

## Data cleaning

In [2]:
import string
import re
def cleaning(text):        
    # converting to lowercase, removing URL links, special characters, punctuations...
    text = text.lower() # converting to lowercase
    text = re.sub(r"\b\d+\b", "", text) # removing number 
    text = re.sub('<.*?>+', '', text) # removing special characters, 
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # punctuations
    text = re.sub('\n', '', text)
    text = re.sub('[’“”…]', '', text)   

    return text
    
df["cleantext"] = df['text'].apply(cleaning)

## Split data -> 80/20 Train/test

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["cleantext"], df["label_id"], test_size=0.2, random_state=42, stratify=df["label_id"]
)



## Logistic Regression with TF-IDF

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=300)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
print(classification_report(y_test, y_pred, target_names=label_map.keys()))


              precision    recall  f1-score   support

    negative       0.88      0.94      0.91       197
     neutral       0.72      0.74      0.73        74
    positive       0.94      0.48      0.64        31

    accuracy                           0.84       302
   macro avg       0.85      0.72      0.76       302
weighted avg       0.85      0.84      0.84       302



## Transformer-based models

In [10]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate

# Build Dataset object
train_df = pd.DataFrame({"text": X_train, "label": y_train})
test_df = pd.DataFrame({"text": X_test, "label": y_test})
train_dataset_full = Dataset.from_pandas(train_df)

# split training data to train/validation 90/10
dataset_split = train_dataset_full.train_test_split(test_size=0.1, seed=42)

# Tokenize
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
train_dataset = dataset_split["train"].map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), 
    batched=True
)

val_dataset = dataset_split["test"].map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), 
    batched=True
)

test_df = pd.DataFrame({"text": X_test.values, "label": y_test.values})
test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), 
    batched=True
)

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Model
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

training_args = TrainingArguments(
    output_dir="./bert_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
)

bert_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

bert_trainer.train()


Map: 100%|██████████| 302/302 [00:00<00:00, 929.90 examples/s]


Train: 1085
Val: 121
Test: 302


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.469900,0.361112,0.876033,0.845554
2,0.338900,0.311021,0.892562,0.855968
3,0.172400,0.302097,0.909091,0.887496
4,0.101900,0.500761,0.900826,0.879526
5,0.078200,0.339298,0.917355,0.882092
6,0.072900,0.449921,0.884298,0.832616
7,0.081900,0.557804,0.876033,0.834659
8,0.035800,0.427272,0.900826,0.849138
9,0.042400,0.450097,0.900826,0.853260
10,0.026400,0.445308,0.900826,0.853260


TrainOutput(global_step=1360, training_loss=0.1470695109490086, metrics={'train_runtime': 352.1361, 'train_samples_per_second': 30.812, 'train_steps_per_second': 3.862, 'total_flos': 1427390291174400.0, 'train_loss': 0.1470695109490086, 'epoch': 10.0})

In [11]:
# Test models
test_results = bert_trainer.evaluate(test_dataset)
print("\nBERT Test metrics:", test_results)

# Get predictions for error analysis
predictions = bert_trainer.predict(test_dataset)
y_pred_bert = predictions.predictions.argmax(-1)
y_true_bert = predictions.label_ids

from sklearn.metrics import classification_report
print("\nDetailed Classification Report:")
print(classification_report(y_true_bert, y_pred_bert, 
                          target_names=['Negative', 'Neutral', 'Positive'],
                          digits=3))


BERT Test metrics: {'eval_loss': 0.6652730703353882, 'eval_accuracy': 0.8642384105960265, 'eval_f1_macro': 0.813726703973324, 'eval_runtime': 2.3718, 'eval_samples_per_second': 127.332, 'eval_steps_per_second': 16.022, 'epoch': 10.0}

Detailed Classification Report:
              precision    recall  f1-score   support

    Negative      0.957     0.893     0.924       197
     Neutral      0.718     0.824     0.767        74
    Positive      0.727     0.774     0.750        31

    accuracy                          0.864       302
   macro avg      0.800     0.831     0.814       302
weighted avg      0.874     0.864     0.868       302



In [12]:
# Get misclassified samples
error_mask = y_pred_bert != y_true_bert
errors_df = pd.DataFrame({
    'text': X_test.values[error_mask],
    'true': y_true_bert[error_mask],
    'pred': y_pred_bert[error_mask],
    'title': df.loc[X_test.index[error_mask], 'title'].values
})

sample_errors = errors_df.sample(10, random_state=42)

label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
for idx, row in enumerate(sample_errors.itertuples(), 1):
    print(f"\n--- Error {idx} ---")
    print(f"True: {label_names[row.true]} | Pred: {label_names[row.pred]}")
    print(f"Title: {row.title}")


--- Error 1 ---
True: Negative | Pred: Neutral
Title: Former CNN 'Journalist' Issues Apology for His Coverage of Joe Biden, But Nobody Is Buying It

--- Error 2 ---
True: Negative | Pred: Neutral
Title: Justin Bieber Once Opened Up About His ‘Turbulent’ Past With Addiction, Admitted It Was An ‘Escape’ For Him, Deets

--- Error 3 ---
True: Positive | Pred: Neutral
Title: “He’s a young man that’s dear to my heart”: Colorado HC Deion Sanders addresses viral interaction with Nate Robinson's son, Nahmier Robinson

--- Error 4 ---
True: Negative | Pred: Positive
Title: Rise of child influencers in Chandigarh: Fun hobby or cause for concern?

--- Error 5 ---
True: Positive | Pred: Neutral
Title: 'Fight Song' singer Rachel Platten answers 7 Questions with Emmy - East Idaho News

--- Error 6 ---
True: Negative | Pred: Positive
Title: Meta teams up with Snap and TikTok to address self-harm content

--- Error 7 ---
True: Neutral | Pred: Positive
Title: Company Offers Rs 41,000 For A Single Stool

## RoBERTa

In [13]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import evaluate


tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [14]:
# Build Dataset object
train_df = pd.DataFrame({"text": X_train.values, "label": y_train.values})

train_dataset_full = Dataset.from_pandas(train_df)
dataset_split = train_dataset_full.train_test_split(test_size=0.1, seed=42)


# Tokenize
train_dataset = dataset_split["train"].map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), 
    batched=True
)
val_dataset = dataset_split["test"].map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), 
    batched=True
)

test_df = pd.DataFrame({"text": X_test.values, "label": y_test.values})
test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(
    lambda e: tokenizer(e["text"], truncation=True, padding="max_length", max_length=256), 
    batched=True
)

print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

training_args = TrainingArguments(
    output_dir="./RoBERTa_sentiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()


Map: 100%|██████████| 302/302 [00:00<00:00, 1632.66 examples/s]


Train: 1085
Val: 121
Test: 302


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.398100,0.304154,0.900826,0.867617
2,0.363800,0.592611,0.867769,0.804121
3,0.204500,0.419169,0.909091,0.873379
4,0.134200,0.705374,0.851240,0.794251
5,0.130000,0.540325,0.909091,0.862507
6,0.046300,0.407637,0.917355,0.869902
7,0.122500,0.538306,0.909091,0.865274
8,0.041200,0.449622,0.917355,0.893243
9,0.048400,0.442874,0.925620,0.899710
10,0.026400,0.447793,0.917355,0.882092


TrainOutput(global_step=1360, training_loss=0.15107704911600142, metrics={'train_runtime': 359.2405, 'train_samples_per_second': 30.203, 'train_steps_per_second': 3.786, 'total_flos': 1427390291174400.0, 'train_loss': 0.15107704911600142, 'epoch': 10.0})

In [15]:
roberta_predictions = trainer.predict(test_dataset)
y_pred_roberta = roberta_predictions.predictions.argmax(-1)
y_true_roberta = roberta_predictions.label_ids

from sklearn.metrics import classification_report
print("\nDetailed Classification Report:")
print(classification_report(y_true_roberta, y_pred_roberta, 
                          target_names=['Negative', 'Neutral', 'Positive'],
                          digits=3))


Detailed Classification Report:
              precision    recall  f1-score   support

    Negative      0.937     0.898     0.917       197
     Neutral      0.744     0.784     0.763        74
    Positive      0.771     0.871     0.818        31

    accuracy                          0.868       302
   macro avg      0.817     0.851     0.833       302
weighted avg      0.872     0.868     0.869       302

